In [1]:
import time
import json
import statistics
from collections import Counter
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium_stealth import stealth

chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(options=chrome_options)

stealth(driver,
        languages=["ru-RU", "ru"],
        vendor="Google Inc.",
        platform="Win32",
        webgl_vendor="Intel Inc.",
        renderer="Intel Iris OpenGL Engine",
        fix_hairline=True)

all_data = []

for page in range(1, 100):
    driver.get(f"https://www.avito.ru/all/avtomobili?p={page}")

    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "[data-marker='item']"))
    )
    time.sleep(2)

    items = driver.find_elements(By.CSS_SELECTOR, "[data-marker='item']")
    for item in items:
        title = item.find_element(By.CSS_SELECTOR, "h3, [data-marker='item-title']").text
        brand = title.split()[0]
        price_text = item.find_element(By.CSS_SELECTOR, "[data-marker='item-price']").text
        price = int(''.join(filter(str.isdigit, price_text)))
        all_data.append({"brand": brand, "price": price})

    print(f"Страница {page}: собрано {len(items)} авто")


brands = [car['brand'] for car in all_data]
brand_counts = Counter(brands)
prices = [car['price'] for car in all_data]
avg_price = sum(prices) / len(prices)
median_price = statistics.median(prices)

print(f"\nВсего авто: {len(all_data)}")
print(f"Средняя цена: {avg_price:,.0f} ₽")
print(f"Медианная цена: {median_price:,.0f} ₽")
print("Топ-10 марок:")
for brand, count in brand_counts.most_common(10):
    print(f"  {brand}: {count} шт. ({count/len(all_data)*100:.1f}%)")

driver.quit()

Страница 99: собрано 50 авто

Всего авто: 4898
Средняя цена: 985,109 ₽
Медианная цена: 650,000 ₽
Топ-10 марок:
  ВАЗ: 1080 шт. (22.0%)
  Hyundai: 370 шт. (7.6%)
  Kia: 345 шт. (7.0%)
  Toyota: 308 шт. (6.3%)
  Volkswagen: 286 шт. (5.8%)
  Chevrolet: 214 шт. (4.4%)
  Ford: 211 шт. (4.3%)
  Renault: 206 шт. (4.2%)
  Nissan: 185 шт. (3.8%)
  BMW: 178 шт. (3.6%)

Данные сохранены в avito_cars.json
